# Green Claims vs Environmental Controversies: A Data Analysis of Fast Fashion Brands

**Student Name:** [Insert Name]  Ke Gao**Student ID:** [Insert ID]  2472610**Module:** ACC102 Mini Assignment, Track 2  \n

This notebook explores whether fast fashion brands with stronger sustainability messaging also face more environmental controversy signals. The analysis uses a small curated educational dataset covering six brands and compares official sustainability messaging with manually curated public-signal controversy counts.

## 2. Problem Definition and User

The analytical problem is to compare sustainability messaging with environmental controversy indicators across selected fast fashion brands. This matters because sustainability claims influence brand reputation, customer trust, and ESG-related investor decisions, yet messaging alone does not necessarily show lower controversy exposure.

The intended users are consumers who want to judge whether sustainability claims look trustworthy, investors who want to identify potential ESG reputation risk, and media or NGOs who may want a simple evidence base for further investigation. Fast fashion is a relevant business context because the sector is highly visible, global, and often discussed in relation to waste, supply chains, and environmental impact.

## 3. Data Sources and Scope

This project uses three small CSV files stored with relative paths. The first file contains curated sustainability messaging texts for Zara, H&M, Shein, Uniqlo, Primark, and Boohoo. These texts are short educational excerpts or summaries based on official corporate sustainability pages and include source metadata and access dates.

The second file contains manually curated environmental controversy counts from 2022 to 2025. These counts are not a validated measure of real environmental harm. They are a public-signal proxy based on a documented search method and should be interpreted cautiously.

The third file provides brand context such as parent group, country, and market segment. The scope is intentionally small to keep the analysis transparent, reproducible, and suitable for a student project.

In [12]:
from pathlib import Path

import pandas as pd

from src.data_prep import (
    aggregate_controversies,
    clean_brand_info,
    clean_controversy_counts,
    clean_sustainability_texts,
    ensure_output_dirs,
    load_raw_data,
    save_dataframe,
)
from src.text_analysis import (
    SUSTAINABILITY_KEYWORDS,
    build_brand_rankings,
    build_merged_metrics,
    build_text_metrics,
)
from src.visualization import generate_all_figures

pd.set_option("display.max_columns", None)
project_dir = Path(".")
ensure_output_dirs(project_dir)

## 4. Data Loading

The project uses `pandas` to load all files with relative paths so the repository remains portable after cloning.

In [13]:
texts_raw, controversy_raw, brand_info_raw = load_raw_data(project_dir)

print("Texts shape:", texts_raw.shape)
print("Controversy shape:", controversy_raw.shape)
print("Brand info shape:", brand_info_raw.shape)

texts_raw.head()

Texts shape: (6, 6)
Controversy shape: (24, 6)
Brand info shape: (6, 4)


,brand,year,source_name,source_url,access_date,text
0,Zara,2024,Inditex sustainability page,https://www.inditex.com/itxcomweb/en/sustainab...,2026-04-23,Our sustainability approach focuses on circula...
1,H&M,2024,H&M Group sustainability page,https://hmgroup.com/sustainability/,2026-04-23,We aim to make fashion more sustainable throug...
2,Shein,2024,SHEIN Group sustainability page,https://www.sheingroup.com/sustainability/,2026-04-23,Our ESG and sustainability programme highlight...
3,Uniqlo,2024,Fast Retailing sustainability page,https://www.fastretailing.com/eng/sustainability/,2026-04-23,Fast Retailing reports sustainability goals co...
4,Primark,2024,Primark Cares page,https://corporate.primark.com/en-gb/primark-cares,2026-04-23,"Primark Cares describes more recycled fibres, ..."


In [14]:
controversy_raw.head()

,brand,year,controversy_news_count,source_note,search_method,access_date
0,Zara,2022,3,Curated public-signal count based on recent en...,Manual search using brand name plus environmen...,2026-04-23
1,Zara,2023,4,Curated public-signal count based on recent en...,Manual search using brand name plus environmen...,2026-04-23
2,Zara,2024,3,Curated public-signal count based on recent en...,Manual search using brand name plus environmen...,2026-04-23
3,Zara,2025,2,Curated public-signal count based on recent en...,Manual search using brand name plus environmen...,2026-04-23
4,H&M,2022,2,Curated public-signal count based on recent en...,Manual search using brand name plus environmen...,2026-04-23


## 5. Data Cleaning and Preparation

The cleaning stage standardizes brand names, checks duplicates, converts year and date fields, prepares cleaned text for keyword analysis, and verifies that the final dataset is usable.

In [15]:
texts = clean_sustainability_texts(texts_raw)
controversies = clean_controversy_counts(controversy_raw)
brand_info = clean_brand_info(brand_info_raw)

print("Duplicate text rows:", texts.duplicated().sum())
print("Duplicate controversy rows:", controversies.duplicated().sum())
print("Missing values in texts:")
print(texts.isna().sum())
print("\nMissing values in controversies:")
print(controversies.isna().sum())

Duplicate text rows: 0
Duplicate controversy rows: 0
Missing values in texts:
brand          0
year           0
source_name    0
source_url     0
access_date    0
text           0
clean_text     0
dtype: int64

Missing values in controversies:
brand                     0
year                      0
controversy_news_count    0
source_note               0
search_method             0
access_date               0
dtype: int64


In [16]:
texts[["brand", "year", "clean_text"]]

,brand,year,clean_text
0,Zara,2024,our sustainability approach focuses on circula...
1,H&M,2024,we aim to make fashion more sustainable throug...
2,Shein,2024,our esg and sustainability programme highlight...
3,Uniqlo,2024,fast retailing reports sustainability goals co...
4,Primark,2024,primark cares describes more recycled fibres d...
5,Boohoo,2024,the group states sustainability priorities aro...


## 6. Text Analysis

A simple keyword-based method is used because it is transparent and easy to explain. The keyword list is designed to capture common sustainability messaging terms: sustainable, sustainability, eco, green, recycled, ethical, carbon, climate, circular, and net zero.

For each brand, the notebook calculates total words, sustainability keyword count, and a sustainability score equal to keyword count divided by total words.

In [17]:
SUSTAINABILITY_KEYWORDS

['sustainable',
 'sustainability',
 'eco',
 'green',
 'recycled',
 'ethical',
 'carbon',
 'climate',
 'circular',
 'net zero']

In [18]:
text_metrics = build_text_metrics(texts)
text_metrics.sort_values("sustainability_score", ascending=False)

,brand,total_words,sustainability_keyword_count,source_count,sustainability_score
4,Uniqlo,19,5,1,0.263158
5,Zara,20,5,1,0.250000
1,H&M,21,5,1,0.238095
2,Primark,22,4,1,0.181818
0,Boohoo,19,3,1,0.157895
3,Shein,20,2,1,0.100000


## 7. Controversy Metrics

The controversy dataset records a manually curated count of public controversy signals by brand and year. These counts are aggregated over the selected period to create one comparable indicator per brand. This is a proxy measure only and should not be treated as definitive proof of misconduct.

In [19]:
controversy_summary = aggregate_controversies(controversies)
controversy_summary

,brand,controversy_news_count,first_year,last_year,period
0,Shein,26,2022,2025,2022-2025
1,Boohoo,20,2022,2025,2022-2025
2,Zara,12,2022,2025,2022-2025
3,H&M,9,2022,2025,2022-2025
4,Primark,8,2022,2025,2022-2025
5,Uniqlo,5,2022,2025,2022-2025


## 8. Merge and Construct Indicators

The text metrics and controversy metrics are merged into one brand-level table. This section constructs a normalized sustainability score, a normalized controversy score, and a `gap_score`, which is defined as normalized sustainability score minus normalized controversy score.

This `gap_score` is an exploratory indicator only. A positive result does not prove better sustainability performance, and a negative result does not prove misconduct. It simply highlights where messaging intensity and controversy exposure look more or less aligned in this curated sample.

In [20]:
merged_metrics = build_merged_metrics(text_metrics, controversy_summary, brand_info)
brand_rankings = build_brand_rankings(merged_metrics)

save_dataframe(text_metrics, project_dir / "data" / "processed" / "text_metrics.csv")
save_dataframe(merged_metrics, project_dir / "data" / "processed" / "merged_brand_metrics.csv")
save_dataframe(brand_rankings, project_dir / "data" / "processed" / "brand_rankings.csv")

brand_rankings[["rank", "brand", "sustainability_score", "controversy_news_count", "gap_score", "interpretation_label"]]

,rank,brand,sustainability_score,controversy_news_count,gap_score,interpretation_label
0,1,Uniqlo,0.263158,5,1.000000,High Messaging / Low Controversy
1,2,H&M,0.238095,9,0.655914,High Messaging / Low Controversy
2,3,Zara,0.250000,12,0.586022,High Messaging / Low Controversy
3,4,Primark,0.181818,8,0.358609,High Messaging / Low Controversy
4,5,Boohoo,0.157895,20,-0.359447,Low Messaging / High Controversy
5,6,Shein,0.100000,26,-1.000000,Low Messaging / High Controversy


## 9. Visualization

The following charts are generated and saved into the `figures/` folder using clear labels and simple visual design.

In [21]:
generate_all_figures(merged_metrics, project_dir / "figures")
print("Figures saved to:", project_dir / "figures")

Figures saved to: figures


## 10. Findings and Interpretation

In this curated dataset, Uniqlo, H&M, and Zara show comparatively stronger sustainability messaging scores than the other selected brands. Shein and Boohoo show the highest controversy proxy counts across the chosen period, while Uniqlo shows the lowest.

The comparison suggests that stronger sustainability messaging does not automatically imply lower controversy exposure. Some brands appear to combine relatively limited messaging scores with higher controversy counts, which may indicate a possible disclosure–performance gap or a possible greenwashing signal worthy of further investigation.

For consumers, this means brand messaging should be read alongside wider public information. For investors, the results show that controversy exposure may remain material even when sustainability language is visible in corporate communication. Because the dataset is small and proxy-based, these findings should be treated as exploratory rather than conclusive.

In [22]:
top_messaging = merged_metrics.sort_values("sustainability_score", ascending=False).head(3)[["brand", "sustainability_score"]]
top_controversy = merged_metrics.sort_values("controversy_news_count", ascending=False).head(3)[["brand", "controversy_news_count"]]

print("Top sustainability messaging brands:")
display(top_messaging)

print("Top controversy proxy brands:")
display(top_controversy)

Top sustainability messaging brands:


,brand,sustainability_score
0,Uniqlo,0.263158
2,Zara,0.250000
1,H&M,0.238095


Top controversy proxy brands:


,brand,controversy_news_count
5,Shein,26
4,Boohoo,20
2,Zara,12


## 11. Limitations

This analysis has several important limitations. The sample size is small, the controversy variable is a proxy based on manually curated public-signal counts, and the sustainability texts are short curated educational excerpts rather than a full archive of company communication. The selected brands and time range are limited, and keyword frequency does not measure real sustainability performance. In addition, controversy counts depend on media visibility, search wording, and the quality of public coverage.

## 12. Conclusion

This notebook asked whether fast fashion brands with stronger sustainability messaging also face more environmental controversies. The results suggest that messaging intensity and controversy exposure do not move together in a simple way. The project provides a practical comparison tool for consumers, investors, and media users, but its conclusions must remain cautious because the dataset is curated and exploratory. Overall, the notebook demonstrates how transparent Python analysis can support a small but meaningful business-focused ESG investigation.